[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Backup and Copying


## What you will be able to do

Copy a SQLite database that programs are still using, in a way that is safe to restore: with
`conn.backup`, with `VACUUM INTO`, as SQL text with `iterdump`, or as bytes with `serialize`.
Explain why a plain copy of the file can catch half of a change, and what a database in WAL mode
keeps outside its file. Check a copy with `PRAGMA integrity_check`, restore from one without
throwing away newer data, and take a backup off a computer whose disk will not last, such as
Colab's.


## The idea

### The problem

A SQLite database is one file, so a backup looks like a copy of a file, and when nothing is writing
to the database, a copy is exactly that. The stations database is rarely so quiet. The loader writes
every hour, and a transaction that changes many readings writes its pages into the file one after
another, sometimes before it commits. A copy taken in the middle holds some new pages and some old
ones: a database that never existed, which SQLite may call damaged, or may open without complaint.

In WAL mode the file is not even the whole database. Every commit since the last checkpoint lives in
the `-wal` file beside it, so a copy of the database file alone leaves out the latest readings, and
can leave out a whole table. Backups made the right way have failures of their own. A scheduled
backup that writes to the same file name every night stops on the second night, a dump taken while
the loader commits mixes two moments, and a restore over the live database throws away every reading
written since the backup. And on a notebook service such as Colab, the files a notebook writes are
deleted when its session ends, along with any backup that was never downloaded.

### What a backup is

> A **backup** of a SQLite database is a copy of it as it really was at one moment: every change
> committed by then, and nothing of any change after. sqlite3 has four ways to make one while the
> database is in use. **`conn.backup(target)`** copies the database page by page into another
> connection's database through SQLite's online backup API, and, copying in batches, starts again
> when another connection changes the source between two of them. **`VACUUM INTO 'file'`** writes a
> new, compacted database file from one consistent read. **`conn.iterdump()`** yields, as text, the
> SQL statements that would build the database again. **`conn.serialize()`** returns the whole
> database as bytes, which **`deserialize`** loads into a connection. **`PRAGMA integrity_check`**
> inspects the structure of a database and returns `ok` when it finds nothing wrong.

### Why it works that way

- **A write changes pages in place, over time.** In the rollback journal mode, a transaction whose
  changes no longer fit in memory writes them into the database file before it commits, keeping the
  old pages in the `-journal` file in case it rolls back. A copy of the database file alone, taken
  meanwhile, holds a mixture, and the journal that would repair it was not copied.
- **WAL spreads a database over two files.** Commits since the last checkpoint are in the `-wal`
  file, so the database file alone lacks them, including any table created since.
- **The backup API copies one committed state.** It reads a batch of pages at a time, and when
  another connection has changed the source since the last batch, it starts over.
- **`VACUUM INTO` rebuilds.** It writes every table and index afresh into a new file, inside one read
  transaction, which leaves out unused pages, and it refuses to write over a file that exists.
- **A dump is text, read a table at a time.** `iterdump` writes SQL that any SQLite can run and a
  person can read, with a query for every table, so only a transaction around it keeps every table to
  the same moment. Restoring a dump runs every `INSERT` again.
- **A restore replaces every page.** Backing a copy up into a database overwrites all of it,
  including rows written after the copy was taken.

### Where this shows up

PostgreSQL, which the **asyncpg and psycopg3, Deep Dive** guide connects to, has `pg_dump`, which
writes a database as SQL as `iterdump` does, and online backups of its files, which is the job
`conn.backup` does here. The **Dagster and Prefect, Deep Dive** and **Apache Airflow, Deep Dive**
guides schedule jobs such as the nightly backup this notebook ends with, and tools such as Litestream
copy the commits in a WAL database's `-wal` file to other storage as they happen. In this guide, the
**Changing a Schema** notebook called a copy the cheapest protection before any change, and the
**A Searchable Archive** notebook finishes by backing up its archive and downloading it.

### What this notebook covers

- A file copied while a transaction writes it, and what the copy holds
- A database in WAL mode, copied with and without its `-wal` file
- `conn.backup`, in batches, while another connection writes
- `VACUUM INTO`, and the smaller file it writes
- `iterdump`, a backup as SQL text, and restoring from it
- `serialize` and `deserialize`, a database as bytes
- Taking a backup off the computer with `files.download` in Colab
- When to use each way of copying
- A nightly backup that checks every copy and keeps the last seven
- Seven errors: a copy without its `-wal` file, `VACUUM INTO` over an existing file, `VACUUM` inside
  a transaction, a dump restored into tables that exist, bytes from a database in WAL mode, a dump
  taken outside a transaction, and a restore over a database in use

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

source = sqlite3.connect(":memory:")
source.execute("CREATE TABLE readings (hour TEXT, celsius REAL)")
source.executemany("INSERT INTO readings VALUES (?, ?)",
                   [(f"2026-01-01T{hour:02d}:00", -3.0) for hour in range(24)])
source.commit()

copy = sqlite3.connect(":memory:")
source.backup(copy)
print("readings in the copy:", copy.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
print("the copy's integrity_check:", copy.execute("PRAGMA integrity_check").fetchone()[0])
print("the same database as SQL:", list(source.iterdump())[:2])
source.close()
copy.close()
```

```
readings in the copy: 24
the copy's integrity_check: ok
the same database as SQL: ['BEGIN TRANSACTION;', 'CREATE TABLE readings (hour TEXT, celsius REAL);']
```

`backup` copied one database into another connection's database, a day of readings with it, and the
copy passed SQLite's own check. `iterdump` described the same database as the SQL that would build it
again, from `BEGIN TRANSACTION;` to the `COMMIT;` at its end.


## Setup

Five imports, and the stations and their year of readings in `stations.db`, in the default rollback
journal mode.

- `sqlite3` builds the database and makes every copy of it
- `math`, `datetime` and `timedelta` make the same year of readings the **Why sqlite3** notebook made
- `Path` names the scratch folder and the files in it
- `shutil` copies files, and removes the scratch folder at the start and at the end

`check` opens a database file on its own connection, and returns how many readings it holds and the
first line of what `PRAGMA integrity_check` reports, which is `ok` when nothing is wrong.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "stations.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}
NEW_READING = "INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)"


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany(NEW_READING, ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def check(path):
    """Open a database file on its own connection, and return its readings and the first line of its integrity_check."""
    conn = sqlite3.connect(path)
    try:
        integrity = conn.execute("PRAGMA integrity_check").fetchone()[0]
        return conn.execute("SELECT COUNT(*) FROM readings").fetchone()[0], integrity
    finally:
        conn.close()


print("built", DATABASE, "| readings and integrity_check:", check(DATABASE))


built scratch/stations.db | readings and integrity_check: (35040, 'ok')


## Worked examples

### A file copied while a transaction writes it

A transaction adds 100 degrees to every reading, on a connection whose page cache holds only 20
pages, so the changed pages go into the file before the commit. The file is copied in the middle,
and then the transaction rolls back:


In [2]:
writer = sqlite3.connect(DATABASE, autocommit=True)
writer.execute("PRAGMA cache_size = 20")
writer.execute("BEGIN")
writer.execute("UPDATE readings SET celsius = celsius + 100")
print("the -journal file exists while the transaction is open:", Path(f"{DATABASE}-journal").exists())
shutil.copy(DATABASE, SCRATCH / "mid_write.db")
writer.execute("ROLLBACK")
writer.close()

caught = sqlite3.connect(SCRATCH / "mid_write.db")
try:
    print("the copy passes integrity_check:", caught.execute("PRAGMA integrity_check").fetchone()[0] == "ok")
    changed, unchanged = caught.execute("SELECT SUM(celsius > 50), SUM(celsius <= 50) FROM readings").fetchone()
    print("the copy holds readings with 100 added, and readings without:", changed > 0 and unchanged > 0)
except sqlite3.DatabaseError as error:
    print("the copy cannot be read:", error)
caught.close()

print("the database itself, after the rollback:", check(DATABASE))
shutil.copy(DATABASE, SCRATCH / "quiet.db")
print("a copy taken with nothing writing:", check(SCRATCH / "quiet.db"))


the -journal file exists while the transaction is open: True
the copy passes integrity_check: False
the copy holds readings with 100 added, and readings without: True
the database itself, after the rollback: (35040, 'ok')
a copy taken with nothing writing: (35040, 'ok')


The copy is half of a change. The transaction had written some of its pages into the file, and the
copy caught those beside pages it had not reached yet, so `integrity_check` found the file's
structure broken, and the readings it could still read are a mixture the database never held. The
database itself was repaired by the rollback, from its `-journal` file, which the copy did not
include. A copy taken once the transaction had ended, with nothing writing, is complete.

### A database in WAL mode, with and without its -wal file

A copy of the database, in WAL mode, with a day of readings for 2026 committed and not yet
checkpointed. The connection, `live`, stays open for the rest of the notebook, so the `-wal` file
stays too:


In [3]:
WAL_DATABASE = SCRATCH / "wal.db"
shutil.copy(DATABASE, WAL_DATABASE)
live = sqlite3.connect(WAL_DATABASE, autocommit=True)
live.execute("PRAGMA journal_mode = WAL")
live.executemany(NEW_READING, [(ids["Oslo"], f"2026-01-01T{hour:02d}:00", -2.0) for hour in range(24)])

shutil.copy(WAL_DATABASE, SCRATCH / "file_only.db")
for suffix in ("", "-wal"):
    shutil.copy(f"{WAL_DATABASE}{suffix}", f"{SCRATCH / 'with_wal.db'}{suffix}")

print("the live database:        ", check(WAL_DATABASE))
print("a copy of the file alone: ", check(SCRATCH / "file_only.db"))
print("a copy with its -wal file:", check(SCRATCH / "with_wal.db"))


the live database:         (35064, 'ok')
a copy of the file alone:  (35040, 'ok')
a copy with its -wal file: (35064, 'ok')


The copy of the file alone passed its check and was missing the day: 35,040 readings where the
database has 35,064, and nothing about it says so. The 24 new readings were in `wal.db-wal`, and the
copy that brought its `-wal` file along had them, since SQLite read the log when it opened the copy.
Copying the two files is still a race, though: a commit or a checkpoint between the two copies leaves
them out of step.

So a copy of the database file is a backup exactly when no connection has the database open and no
`-journal` or `-wal` file lies beside it, which is how the last connection to close normally leaves
a database. That is the condition the **Why sqlite3** notebook promised to make exact. At any other
time, copy the database through SQLite, in one of the ways that follow.

### conn.backup

`backup` copies the database it is called on into the target connection's database. With `pages`, it
copies that many pages at a time, and calls `progress` after every batch with a status, the pages
left and the pages in all. Here another connection commits a reading after the first batch:


In [4]:
writer = sqlite3.connect(WAL_DATABASE, autocommit=True)
left_after_each_batch = []


def progress(status, remaining, total):
    """Record the pages left after a batch, and have the writer commit a reading after the first."""
    left_after_each_batch.append(remaining)
    if len(left_after_each_batch) == 1:
        writer.execute(NEW_READING, (ids["Bergen"], "2026-01-01T00:00", 4.4))


target = sqlite3.connect(SCRATCH / "backup.db")
live.backup(target, pages=50, progress=progress)
target.close()
writer.close()

print("copied in batches:", len(left_after_each_batch) > 1)
print("the second batch started again from the first page:", left_after_each_batch[1] >= left_after_each_batch[0])
print("the live database:", check(WAL_DATABASE))
print("the backup:       ", check(SCRATCH / "backup.db"))


copied in batches: True
the second batch started again from the first page: True
the live database: (35065, 'ok')
the backup:        (35065, 'ok')


After the first batch, another connection committed a reading, so the second batch found the source
changed and started over, and the pages left did not go down. The backup holds 35,065 readings, the
new one among them, and passed its check: it is the database as it was after that commit. A backup
never catches half of a change, and a backup of a busy database may start over many times. Between
batches, other connections are free to write, which is what `pages` is for; with the default,
`pages=-1`, the whole copy is one step.

One case never finishes: a connection backing up its own database while it has a write transaction
open. SQLite reports the source as busy until that transaction ends, `backup` tries again until it
does, and in the same thread it never will, so commit first.

### VACUUM INTO

`VACUUM INTO` writes a new database file from one consistent read, rebuilding every table, and never
writes over a file that already exists. Bergen's and Oslo's readings are deleted first, and a
checkpoint writes the deletions into the live file, which keeps its size, with the space marked free:


In [5]:
live.execute("DELETE FROM readings WHERE station_id IN (?, ?)", (ids["Bergen"], ids["Oslo"]))
live.execute("PRAGMA wal_checkpoint(TRUNCATE)")
live.execute("VACUUM INTO ?", (str(SCRATCH / "vacuumed.db"),))

print("the live database:", check(WAL_DATABASE))
print("the new file:     ", check(SCRATCH / "vacuumed.db"))
print("the new file is smaller:", (SCRATCH / "vacuumed.db").stat().st_size < WAL_DATABASE.stat().st_size)
vacuumed = sqlite3.connect(SCRATCH / "vacuumed.db")
print("the new file's journal_mode:", vacuumed.execute("PRAGMA journal_mode").fetchone()[0])
vacuumed.close()


the live database: (17520, 'ok')
the new file:      (17520, 'ok')
the new file is smaller: True
the new file's journal_mode: delete


The new file holds every reading the live database holds, without the pages the deleted readings
left free. It is also in the rollback journal mode, where `backup` kept the source's WAL mode, since
`VACUUM INTO` builds a new database rather than copying pages, so a program that puts this copy back
in place has to set WAL mode again. The file's name goes in through a placeholder, as any value
does.

### iterdump: a backup as SQL text

`iterdump` yields the statements that would build the database again, one at a time, so a large
database can be written to a file without being held in memory. It reads the database with a query
for every table, so it runs inside a transaction here, which keeps every table at the same moment:


In [6]:
DUMP = SCRATCH / "stations.sql"
live.execute("BEGIN")
with DUMP.open("w", encoding="utf-8") as file:
    for statement in live.iterdump():
        file.write(statement + "\n")
live.execute("COMMIT")

lines = DUMP.read_text(encoding="utf-8").splitlines()
for line in lines[:4] + ["..."] + lines[-2:]:
    print(line)

restored = sqlite3.connect(SCRATCH / "restored.db")
restored.executescript(DUMP.read_text(encoding="utf-8"))
restored.close()
print("restored from the dump:", check(SCRATCH / "restored.db"))


BEGIN TRANSACTION;
CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
INSERT INTO "readings" VALUES(3,3,'2025-01-01T00:00',-14.6);
...
INSERT INTO "stations" VALUES(5,'Kirkenes',69.73);
COMMIT;
restored from the dump: (17520, 'ok')


The dump is a transaction of its own: `BEGIN TRANSACTION;`, a `CREATE TABLE` for every table in
alphabetical order, exactly as the table was created, line break and all, an `INSERT` for every row,
and `COMMIT;`. `executescript` ran all of it into a new database, which passed its check. A dump
suits a small database, a copy people should be able to read or keep in version control, or a
database bound for another system, where the SQL can be edited on the way. Restoring it runs every
`INSERT` again, so for a large database it is by far the slowest way back.

### serialize and deserialize: a database as bytes

`serialize` returns the database as the bytes its file would hold, and `deserialize` makes a
connection's database those bytes, which suits a database in memory, such as one sent over a network
or one built once and loaded fresh for every test:


In [7]:
source = sqlite3.connect(DATABASE)
data = source.serialize()
source.close()
print("the bytes begin:", data[:16])

in_memory = sqlite3.connect(":memory:")
in_memory.deserialize(data)
print("readings, in memory:", in_memory.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
in_memory.close()


the bytes begin: b'SQLite format 3\x00'
readings, in memory: 35040


The bytes begin with `SQLite format 3` and a zero byte, the header of every SQLite database file,
which the **Why sqlite3** notebook read from `stations.db`. The connection in memory then held the
whole year. Both need the whole database in memory, twice over for a moment, so they suit small
databases, and the bytes must come from a database that is not in WAL mode, as the Common errors
show.

### Taking a backup off the computer

On Colab, the files a notebook writes disappear when its session ends. `files.download`, from
`google.colab`, sends a file to the browser, and the import fails anywhere but Colab, where the file
is already on a disk that stays:


In [8]:
try:
    from google.colab import files
except ImportError:
    print("not running in Colab, so the backup stays at", SCRATCH / "backup.db")
else:
    files.download(str(SCRATCH / "backup.db"))


not running in Colab, so the backup stays at scratch/backup.db


In Colab, the browser saves `backup.db` to your computer. A backup on the same disk as its database
protects against a mistake, not against losing the disk, so a backup worth the name ends up somewhere
else as well.

### Which way to copy

| Write | When | Why |
|---|---|---|
| `shutil.copy` of the database file | no connection has the database open, and no `-journal` or `-wal` file lies beside it | the file is then the whole database, and a copy of it is the fastest backup |
| `conn.backup(target)` | a database in use, copied into another connection, or a copy that must keep WAL mode | it copies pages from one committed state, starts over when another connection writes, and keeps WAL mode |
| `VACUUM INTO 'file'` | a database in use, copied to a new file | one consistent read, no unused pages, and it never writes over a file |
| `conn.iterdump()`, inside a transaction | a copy people can read, keep in version control, or load into another system | plain SQL, independent of SQLite's file format, and slow to restore |
| `conn.serialize()` | a small database that is not in WAL mode, moved as bytes or loaded into memory for tests | the whole database as one `bytes` object, loaded back with `deserialize` |

The default for a database in use is `VACUUM INTO` a new file with the date in its name, checked with
`integrity_check`. The same copy, taken just before a migration, is the protection the
**Changing a Schema** notebook recommended. Use `conn.backup` when the copy goes straight into
another connection, or has to keep WAL mode.

### A nightly backup that keeps the last seven

The pieces of this notebook in one job. For ten nights, the loader adds a day of readings to the
live database, and `back_up` writes that night's copy with `VACUUM INTO` under a name that marks it
unchecked, checks it, and only then gives it the night's name, doing nothing if a backup for the
night exists already. `keep_latest` then deletes all but the newest seven:


In [9]:
BACKUPS = SCRATCH / "backups"
BACKUPS.mkdir()


def back_up(database, folder, night):
    """Copy a database in use to folder/stations-<night>.db, unless that night's backup exists, checking the copy first."""
    target = folder / f"stations-{night}.db"
    if target.exists():
        return target
    unchecked = folder / f"stations-{night}.unchecked"
    unchecked.unlink(missing_ok=True)
    source = sqlite3.connect(database, autocommit=True)
    try:
        source.execute("VACUUM INTO ?", (str(unchecked),))
    finally:
        source.close()
    _, integrity = check(unchecked)
    if integrity != "ok":
        raise RuntimeError(f"the backup for {night} failed its check: {integrity}")
    return unchecked.rename(target)


def keep_latest(folder, count):
    """Delete all but the newest `count` backups in folder."""
    for old in sorted(folder.glob("stations-*.db"))[:-count]:
        old.unlink()


for night in range(1, 11):
    live.executemany(NEW_READING, [(ids["Tromso"], f"2026-01-{night:02d}T{hour:02d}:00", -6.0) for hour in range(24)])
    back_up(WAL_DATABASE, BACKUPS, f"2026-01-{night:02d}")
    keep_latest(BACKUPS, 7)
back_up(WAL_DATABASE, BACKUPS, "2026-01-10")

names = sorted(path.name for path in BACKUPS.iterdir())
print(len(names), "files, from", names[0], "to", names[-1])
print("the live database:", check(WAL_DATABASE))
print("the newest backup:", check(BACKUPS / "stations-2026-01-10.db"))
print("the oldest backup:", check(BACKUPS / "stations-2026-01-04.db"))


7 files, from stations-2026-01-04.db to stations-2026-01-10.db
the live database: (17760, 'ok')
the newest backup: (17760, 'ok')
the oldest backup: (17616, 'ok')


Ten nights made ten backups, and seven remain: the newest holds every reading the live database
holds, and the oldest is six nights of readings behind it. A copy became a backup only by the rename
after its check, so a file with a backup's name is always a finished, checked backup, and a copy cut
short by a crash leaves only an unchecked file, which the next run deletes. That is also why running
the tenth night again did nothing. The dates sort as text in the order they happened, which is why
`keep_latest` can sort the names.

### Where each part came from

| In the nightly backup | What it relies on | The section that showed it |
|---|---|---|
| `VACUUM INTO ?` on a database in use | one consistent read, written to a new file | VACUUM INTO |
| `unchecked.unlink(missing_ok=True)` first | `VACUUM INTO` never writes over a file | VACUUM INTO |
| `check(unchecked)` before the rename | `integrity_check` on the copy, which a copy caught mid-write can fail | A file copied while a transaction writes it |
| a name with the date in it | backups that sort by name, and never replace one another | Which way to copy |
| `live` writing in WAL mode between nights | a copy made through SQLite includes the commits still in the `-wal` file | A database in WAL mode, with and without its -wal file |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/18-backup-and-copying-solutions.ipynb).

**1.** Back up `stations.db` into a new file with `conn.backup`, 40 pages at a time, count the
batches with a `progress` function, and check the copy.


In [10]:
# your code here


**2.** Take a `VACUUM INTO` copy of `stations.db` named `stations-2026-02-01.db`, try again with the
same name, and handle the refusal by adding `-2` to the name.


In [11]:
# your code here


**3.** Dump `stations.db` with `iterdump` inside a transaction, restore the dump into a new database,
and show that the sum of every temperature is the same in both.


In [12]:
# your code here


**4.** Load `stations.db` into memory with `serialize` and `deserialize`, delete Svalbard's readings
there, and show that the file still has all of them.


In [13]:
# your code here


**5.** Put a copy of `stations.db` into WAL mode, copy it with `conn.backup` and with `VACUUM INTO`,
and show which of the two copies is in WAL mode.


In [14]:
# your code here


**6.** Take a backup of a copy of `stations.db`, add a reading to that copy, then restore the backup
into a new file and compare how many readings the two have.


In [15]:
# your code here


## Common errors

### sqlite3.OperationalError: no such table: notes


In [16]:
live.execute("CREATE TABLE notes (station_id INTEGER, note TEXT)")
live.execute("INSERT INTO notes VALUES (?, ?)", (ids["Tromso"], "Heater checked."))
shutil.copy(WAL_DATABASE, SCRATCH / "no_notes.db")

copied = sqlite3.connect(SCRATCH / "no_notes.db")
copied.execute("SELECT * FROM notes").fetchall()


OperationalError: no such table: notes

The table was created after the last checkpoint, so its `CREATE TABLE` and its first row were only
in `wal.db-wal`, and the copy of the database file alone has no table called `notes`. Earlier, the
file alone was only missing readings, and nothing failed at all. Copy a database in use through
SQLite, with `backup` or `VACUUM INTO`, which read the `-wal` file along with the database file:


In [17]:
copied.close()
live.execute("VACUUM INTO ?", (str(SCRATCH / "with_notes.db"),))
copied = sqlite3.connect(SCRATCH / "with_notes.db")
print(copied.execute("SELECT * FROM notes").fetchall())
copied.close()


[(4, 'Heater checked.')]


### sqlite3.OperationalError: output file already exists


In [18]:
live.execute("VACUUM INTO ?", (str(SCRATCH / "with_notes.db"),))


OperationalError: output file already exists

`VACUUM INTO` never writes over a file, which protects the last backup from a job run twice by
mistake. Give every backup a name of its own, as the nightly backup did with the date, or delete the
old file on purpose first:


In [19]:
live.execute("VACUUM INTO ?", (str(SCRATCH / "with_notes-2.db"),))
print("the second copy:", check(SCRATCH / "with_notes-2.db"))


the second copy: (17760, 'ok')


### sqlite3.OperationalError: cannot VACUUM from within a transaction


In [20]:
loader = sqlite3.connect(WAL_DATABASE)
loader.execute(NEW_READING, (ids["Svalbard"], "2026-01-11T00:00", -14.0))
loader.execute("VACUUM INTO ?", (str(SCRATCH / "after_the_load.db"),))


OperationalError: cannot VACUUM from within a transaction

The loader uses sqlite3's default transaction control, so its `INSERT` began a transaction that is
still open, and no kind of `VACUUM` can run inside one. Finish the load first, with a commit or a
rollback, then take the copy:


In [21]:
loader.commit()
loader.execute("VACUUM INTO ?", (str(SCRATCH / "after_the_load.db"),))
print("the copy after the load:", check(SCRATCH / "after_the_load.db"))
loader.close()


the copy after the load: (17761, 'ok')


### sqlite3.OperationalError: table readings already exists


In [22]:
target = sqlite3.connect(SCRATCH / "not_empty.db")
target.execute("CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER, hour TEXT, celsius REAL)")
target.executescript(DUMP.read_text(encoding="utf-8"))


OperationalError: table readings already exists

A dump creates every table it holds, and this database already had a `readings` table, so the script
stopped at that `CREATE TABLE`, inside the dump's own `BEGIN TRANSACTION;`, which is still open. Roll
that back, and restore a dump into a new, empty database:


In [23]:
print("left in a transaction:", target.in_transaction)
target.rollback()
target.close()

fresh = sqlite3.connect(SCRATCH / "fresh_restore.db")
fresh.executescript(DUMP.read_text(encoding="utf-8"))
fresh.close()
print("restored into an empty database:", check(SCRATCH / "fresh_restore.db"))


left in a transaction: True
restored into an empty database: (17520, 'ok')


### sqlite3.OperationalError: unable to open database file


In [24]:
in_memory = sqlite3.connect(":memory:")
in_memory.deserialize(live.serialize())
in_memory.execute("SELECT COUNT(*) FROM readings").fetchone()


OperationalError: unable to open database file

`live` is in WAL mode, which a database records in its file header, so the bytes said so too. A
database loaded from bytes cannot use WAL, so SQLite could not open it as the bytes described it, and
its error, the general one for a file it cannot open, says nothing about WAL. Serialize a copy in the
rollback journal mode, such as one written by `VACUUM INTO`:


In [25]:
in_memory.close()
with_notes = sqlite3.connect(SCRATCH / "with_notes.db")
in_memory = sqlite3.connect(":memory:")
in_memory.deserialize(with_notes.serialize())
with_notes.close()
print("readings, in memory:", in_memory.execute("SELECT COUNT(*) FROM readings").fetchone()[0])
in_memory.close()


readings, in memory: 17760


### No error, and a dump from two moments: iterdump run outside a transaction


In [26]:
loader = sqlite3.connect(WAL_DATABASE, autocommit=True)
BY_NAME = "SELECT name, (SELECT COUNT(*) FROM readings WHERE station_id = stations.id) FROM stations WHERE name = ?"


def dump_while_loading(conn, station, latitude):
    """Dump a database into memory, while the loader adds a station and its first reading in one transaction."""
    statements = []
    for statement in conn.iterdump():
        statements.append(statement)
        if len(statements) == 100:
            loader.execute("BEGIN")
            station_id = loader.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (station, latitude)).lastrowid
            loader.execute(NEW_READING, (station_id, "2026-01-12T00:00", -9.0))
            loader.execute("COMMIT")
    restored = sqlite3.connect(":memory:")
    restored.executescript("\n".join(statements))
    return restored


restored = dump_while_loading(live, "Vardo", 70.37)
print("the live database:", live.execute(BY_NAME, ("Vardo",)).fetchall())
print("the dump:         ", restored.execute(BY_NAME, ("Vardo",)).fetchall())
restored.close()


the live database: [('Vardo', 1)]
the dump:          [('Vardo', 0)]


The dump read `readings` before the loader's transaction and `stations` after it, so it holds a
station that, in the database, never existed without its first reading. `iterdump` runs a query for
every table, and outside a transaction every query sees the database as it is when that query
starts. Run it inside a transaction, which holds one moment for all of them:


In [27]:
live.execute("BEGIN")
restored = dump_while_loading(live, "Hammerfest", 70.66)
live.execute("COMMIT")
print("the live database:", live.execute(BY_NAME, ("Hammerfest",)).fetchall())
print("the dump:         ", restored.execute(BY_NAME, ("Hammerfest",)).fetchall())
restored.close()
loader.close()


the live database: [('Hammerfest', 1)]
the dump:          []


The dump is the database as it was when the transaction began, before the loader added Hammerfest, so
the station and its reading are either both in a dump or both out of it.

### No error, and today's readings gone: a backup restored over a database in use


In [28]:
TABLES = "SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name"
print("before the restore:", check(WAL_DATABASE), [name for (name,) in live.execute(TABLES)])
last_night = sqlite3.connect(BACKUPS / "stations-2026-01-10.db")
last_night.backup(live)
last_night.close()
print("after the restore: ", check(WAL_DATABASE), [name for (name,) in live.execute(TABLES)])


before the restore: (17763, 'ok') ['notes', 'readings', 'stations']
after the restore:  (17760, 'ok') ['readings', 'stations']


The restore copied every page of the backup over the live database, which now holds exactly what the
backup held. Everything written since the backup went with it: the Svalbard reading, the two new
stations and their readings, and the `notes` table, and no connection was warned. In a real database,
those rows would be gone for good. Restore a backup into a new file instead, beside the database in
use, where it can be read and compared, and rows can be copied back from it with ordinary SQL:


In [29]:
night_9 = sqlite3.connect(BACKUPS / "stations-2026-01-09.db")
beside = sqlite3.connect(SCRATCH / "as_of_9_january.db")
night_9.backup(beside)
night_9.close()
print("the backup, restored beside it:", beside.execute("SELECT COUNT(*) FROM readings").fetchone()[0], "readings")
print("the live database, untouched:  ", live.execute("SELECT COUNT(*) FROM readings").fetchone()[0], "readings")
beside.close()


the backup, restored beside it: 17736 readings
the live database, untouched:   17760 readings


Last, this cell closes `live`, the last connection still open, and removes the scratch folder, with
every database and copy in it:


In [30]:
live.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A copy of a database file is a backup only when no connection has the database open and no
  `-journal` or `-wal` file lies beside it. A copy taken during a transaction can hold half of a
  change.
- In WAL mode, the latest commits are in the `-wal` file, so a copy of the database file alone can
  lack readings, or a whole table, without any error.
- `conn.backup(target)` copies a database in use, in batches with `pages`, starts over when another
  connection writes, and keeps WAL mode. It never finishes while its own connection has a write
  transaction open.
- `VACUUM INTO 'file'` writes a compacted copy from one consistent read, in the rollback journal
  mode. It never writes over a file, and cannot run inside a transaction.
- `iterdump` writes a database as SQL, of one moment only inside a transaction, and a dump is
  restored with `executescript` into an empty database.
- `serialize` and `deserialize` move a small database as bytes, which must come from a database that
  is not in WAL mode.
- Check every backup before trusting it, give every backup a name of its own, restore into a new file
  rather than over a database in use, and keep backups somewhere other than the computer the database
  is on.


## What is next

The **A Searchable Archive** notebook puts this guide together: a folder of documents loaded into one
database file, searched with FTS5, its lookups checked with `EXPLAIN QUERY PLAN`, its schema changed
by a migration recorded in `user_version`, and the whole file backed up and downloaded at the end.


---

&#8592; **Previous:** [Concurrency and WAL](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/17-concurrency-and-wal.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [A Searchable Archive](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/19-a-searchable-archive.ipynb) &#8594;
